# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Part 1: The Data Contract

**1. What does one row mean, for my lane specifically?**
One row = one content page (identified by `content_id` in the starter
dataset, `content_hash_id` in the warehouse), measured over a trailing
90-day window. Each row captures that page's search performance
(impressions, clicks, CTR, position) and engagement performance
(sessions, engagement rate, scroll rate) for that window, along with
content metadata (type, intent, word count, age, freshness).

**2. Which table(s) will I use?**
For W01–W02, I used the starter dataset:
`data/raw/content_refresh_anonymized.csv`. For this warehouse check,
I'm using `dim_content` joined to `fact_content_daily_performance`
(aliased `fact_daily`), on `content_hash_id`.

**3. What time window am I working in?**
90 days, trailing — matching the starter dataset's `*_90d` columns
(`impressions_90d`, `clicks_90d`, `sessions_90d`, etc.). For the
warehouse check, I used March 2026 as a mid-panel proxy for a 90-day
window. This is wide enough to smooth out day-to-day noise while still
reflecting recent performance rather than very old history. I'm not
yet building a future-looking window (e.g., "next 30 days") because my
current label is a present-moment rule-based gap, not a predicted
future outcome (see #4 below).

**4. What would I predict or rank (the label, or a proxy for it)?**
I would rank pages in order of how much attention they need —
specifically, by the size of the gap between a page's actual CTR (or
engagement rate) and the expected CTR (or engagement rate) for its
position tier, among pages with enough volume to trust the number. A
bigger gap means more attention needed. This is a defined-rule proxy,
not an observed future outcome — I'm comparing each page against its
peers right now, not checking whether its performance actually changed
afterward.

**5. What is one thing I'm deliberately excluding?**
I am deliberately excluding `ai_sessions_90d` and `ai_traffic_pct` from
my slice. These columns belong to the AI Referral Opportunity lane,
and the lane guide explicitly flags this data as sparse (roughly
30,000 rows with any AI-session signal, out of tens of millions of
total daily rows in the full warehouse). Including them here would
risk drawing conclusions from a signal that's too thin to support
them, and it's outside my lane's actual question anyway.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Part 2: Sorting Fields Into Buckets

**Features** (inputs my scoring will actually use):
- `impressions_90d` — trailing search volume, needed to judge whether
  a CTR number is trustworthy
- `avg_position` — trailing average search rank, needed to build
  position tiers
- `main_intent` — content classification, may explain why CTR differs
  even within the same position tier
- `word_count` — static content property, useful context for thin-
  content review candidates
- `content_age_days` — how old the page is, useful for freshness-
  related reason codes

**Label / proxy** (what I'm ranking by — not a raw column, something
I compute):
- The gap between a page's actual CTR (or engagement rate) and the
  expected CTR (or engagement rate) for its position tier.

**Context** (useful for interpretation and reason codes, not fed to
the model/score directly):
- `content_type`, `freshness_tier`, `trend_direction`, `trend_pct`

**Excluded** (and why):
- `ai_sessions_90d`, `ai_traffic_pct` — sparse data belonging to a
  different lane (see Part 1, #5)
- `search_volume`, `competition`, `competition_level`, `cpc` — keyword-
  economics fields, not page-performance signals
- `provider_used`, `model_used` — these appear to be pipeline/data-
  generation metadata (how the dataset itself was built), not observed
  page-performance signals, so they should never be treated as
  features
- `impressions_last_30d` / `_prev_30d` and equivalents — these are
  future-window building blocks for a different kind of label (Growth/
  Recovery lane), not needed for this present-moment gap

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
#QUERY1
grain_total = con.sql(f"""
    SELECT COUNT(*) AS total_rows FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    )
""").df()

grain_distinct = con.sql(f"""
    SELECT COUNT(*) AS unique_combos FROM (
        SELECT DISTINCT report_date, client_hash_id, content_hash_id
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    )
""").df()

print(grain_total, grain_distinct)



#QUERY2
size_span_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

size_span_check

#QUERY3
availability_check = con.sql(f"""
    SELECT COUNT(*) AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND ga4_data_available IS TRUE
""").df()

availability_check

#QUERY1
**Grain check:** Total rows in the March slice exactly equal the count
of distinct (report_date, client_hash_id, content_hash_id)
combinations. This confirms the table's documented grain holds true —
one row genuinely represents one client-content-day, with no
duplicates.

#QUERY2
**Size/span check:** The March slice contains 9,841,378 rows, and the
min/max report_date values both fall within March 2026 — confirming
the date filter is correctly bounded. This is roughly 12.5% of the
full ~78.8M-row table for a single month, a reasonable, usable slice
size to prototype against.

#QUERY3
**Availability check:** Of the 9,841,378 total March rows, only
413,966 (about 4.2%) have `ga4_data_available = TRUE`. This means
engagement-side signals (sessions, engagement_rate, scroll_rate) are
only usable for a small fraction of March's data — the remaining ~96%
of rows are search-only (GSC data present, no GA4 tracking yet for
that client at that date).

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.